In [1]:
import torch
from PIL import Image
import open_clip
import pandas as pd
import numpy as np

In [2]:
model, _, preprocess = open_clip.create_model_and_transforms('ViT-B-16', 
                                                             pretrained='pathgenclip_models/pathgenclip_best_breast.pt') 
model.eval()  
tokenizer = open_clip.get_tokenizer('ViT-B-16')

In [3]:
queries = []

queries.append(" Comparison with a normal breast section shows presence of ductal and lobular structures in the fibro-adipose stroma. if we zoom in a little bit we can see some irregular nests of cells within the fibrous stroma. So actually the original ductal and lobular architecture of the breast is lost here. Just for comparison, here is a section taken from a normal breast and again you can see the fibro-adipose stroma and here are the ductal and the lobular structures. So this architecture is not seen in the core biopsy.")
queries.append("Clusters of cells called lobules are present, which contain ducts and glandular tissue or acini in an active mammary gland. by a vast network of dense, irregular connective tissue. And within it, we can see patches of adipocytes, or adipose tissue. And also, we can see these clusters of cells. And these clusters of cells are important because these are what are called lobules. And within each one of these lobules, we're actually going to find a series of ducts. And in an active mammary gland, we'd see a set of ducts as well as glandular tissue or acini.")
queries.append(" Presence of fissured and cracked amorphous pink material in the dermis associated with red cell extravasation, suggestive of amyloid deposition. Thioflavin-T or congo red stains are better for immunoglobulin derived amyloid. Colloid milium stains weakly for amyloid and has a background of solar elastosis. okay Yeah, that is fissured and cracked. So fissured amorphous pink material and cracked sitting in the dermis associated with red cell extravasation. So amyloid would be absolutely your thought. This was a patient with Waldenstroms and deposition, but amyloid, thioflavin T would be a great stain")
 

In [4]:
image = preprocess(Image.open("brst_lbls.jpg")).unsqueeze(0)

text = tokenizer(queries)

with torch.no_grad(), torch.cuda.amp.autocast():
    image_features = model.encode_image(image)
    text_features = model.encode_text(text)
    image_features /= image_features.norm(dim=-1, keepdim=True)
    text_features /= text_features.norm(dim=-1, keepdim=True)

    text_probs = (100.0 * image_features @ text_features.T).softmax(dim=-1)

print("Label probs:", text_probs)

C:\Users\Imroze\AppData\Local\Temp\ipykernel_34400\3365543100.py:5: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with torch.no_grad(), torch.cuda.amp.autocast():


Label probs: tensor([[1.7990e-05, 9.9998e-01, 5.5802e-09]])
